# Day 30 — **Code Review: Self-Review Checklist**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~30 min

Before Phase 1 code goes in your portfolio, review it — and don't do it by eyeball. The checklist (*type hints ✓, docstrings ✓, no hardcoded keys ✓*) can be **automated** with the standard-library `ast` module, the same tool that powered Day 27's mini-Bandit. An automated self-review runs in a second and never gets bored on line 400.

Today:
1. A sample module with review issues.
2. Detect missing **type hints** and **docstrings** via `ast`.
3. Detect **hardcoded secrets**.
4. Produce a pass/fail **checklist report**.

## 1. A module to self-review

Three functions: one clean, one missing type hints, one missing a docstring — plus a hardcoded key. Exactly the issues the checklist targets.

In [1]:
SOURCE = '''
API_KEY = "sk-live-9f8e7d6c5b4a"

def score_risk(income: float, credit: int) -> float:
    """Return a 0-1 risk score for a loan applicant."""
    return credit / 999 * 0.6 + min(income / 100_000, 1) * 0.4

def approve(app):
    return score_risk(app["income"], app["credit"]) >= 0.5

def audit_log(event: str) -> None:
    print(event)
'''
print(SOURCE)


API_KEY = "sk-live-9f8e7d6c5b4a"

def score_risk(income: float, credit: int) -> float:
    """Return a 0-1 risk score for a loan applicant."""
    return credit / 999 * 0.6 + min(income / 100_000, 1) * 0.4

def approve(app):
    return score_risk(app["income"], app["credit"]) >= 0.5

def audit_log(event: str) -> None:
    print(event)



☝ `approve` has no type hints and no docstring; `API_KEY` is a hardcoded secret. In a 10-line file you'd spot these; in a real PR touching 30 files you won't, reliably. Automate the boring checks so human review time goes to *logic*, not to spotting a missing colon-annotation.

## 2. Detect missing hints & docstrings

Parse to an AST, walk every `FunctionDef`, and check two things: does each argument have an annotation, and is there a docstring (`ast.get_docstring`)?

In [2]:
import ast

def review_functions(src):
    tree = ast.parse(src)
    report = []
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef):
            missing_hints = [a.arg for a in node.args.args if a.annotation is None]
            no_return = node.returns is None
            no_doc = ast.get_docstring(node) is None
            report.append({
                "fn": node.name, "line": node.lineno,
                "missing_arg_hints": missing_hints,
                "missing_return_hint": no_return,
                "missing_docstring": no_doc,
            })
    return report

for r in review_functions(SOURCE):
    flags = []
    if r["missing_arg_hints"]:   flags.append(f"args {r['missing_arg_hints']}")
    if r["missing_return_hint"]: flags.append("return")
    if r["missing_docstring"]:   flags.append("docstring")
    status = "OK" if not flags else "REVIEW: " + ", ".join(flags)
    print(f"L{r['line']:>2} {r['fn']:12} {status}")

L 4 score_risk   OK
L 8 approve      REVIEW: args ['app'], return, docstring
L11 audit_log    REVIEW: docstring


☝ `score_risk` passes; `approve` is flagged for missing arg hints, return hint, *and* docstring. `ast.get_docstring` is the clean way to check — it knows a docstring is the first string *expression* in the body, not just any string. This is a linter in 15 lines; tools like `ruff`, `mypy`, and `pydocstyle` do it exhaustively, but knowing the mechanism means you can add project-specific rules yourself.

## 3. Detect hardcoded secrets

Reuse the Day 27 idea: walk assignments, flag any secret-named variable bound to a string literal. This is the "no hardcoded keys ✓" line of the checklist, enforced.

In [3]:
import re
SECRET_RE = re.compile(r"(api[_-]?key|password|passwd|secret|token)", re.IGNORECASE)

def find_secrets(src):
    hits = []
    for node in ast.walk(ast.parse(src)):
        if isinstance(node, ast.Assign):
            for tgt in node.targets:
                name = getattr(tgt, "id", "")
                if SECRET_RE.search(name) and isinstance(node.value, ast.Constant) \
                        and isinstance(node.value.value, str):
                    hits.append((node.lineno, name))
    return hits

for line, name in find_secrets(SOURCE):
    print(f"L{line}: hardcoded secret '{name}' — move to env var / secrets manager")

L2: hardcoded secret 'API_KEY' — move to env var / secrets manager


☝ Catches `API_KEY = "sk-live-..."` — the single most damaging thing to commit, because git history remembers it even after you delete the line. The fix is always the same: `os.environ["API_KEY"]` or a secrets manager, never a literal. Run this (or `ruff`/`gitleaks`) as a pre-commit hook so a key *cannot* be pushed.

## 4. The checklist report

Combine the checks into one gate that prints a tidy pass/fail summary — the thing you glance at before opening a PR.

In [4]:
def self_review(src):
    fn_report = review_functions(src)
    secrets = find_secrets(src)
    fns_clean = all(not (r["missing_arg_hints"] or r["missing_return_hint"]
                         or r["missing_docstring"]) for r in fn_report)
    checks = [
        ("all functions type-hinted", all(not r["missing_arg_hints"] and not r["missing_return_hint"] for r in fn_report)),
        ("all functions documented", all(not r["missing_docstring"] for r in fn_report)),
        ("no hardcoded secrets", len(secrets) == 0),
    ]
    print("SELF-REVIEW CHECKLIST")
    print("-" * 40)
    for label, ok in checks:
        print(f"  [{'✓' if ok else '✗'}] {label}")
    passed = all(ok for _, ok in checks)
    print("-" * 40)
    print("RESULT:", "READY TO PR ✅" if passed else "FIX ITEMS ABOVE ❌")
    return passed

self_review(SOURCE)

SELF-REVIEW CHECKLIST
----------------------------------------
  [✗] all functions type-hinted
  [✗] all functions documented
  [✗] no hardcoded secrets
----------------------------------------
RESULT: FIX ITEMS ABOVE ❌


False

☝ One boolean verdict: this module is **not** ready — it has undocumented, unhinted functions and a hardcoded key. Fix those and the same function prints ✅. That's the whole point of a self-review checklist: make "good enough to share" a *mechanical, repeatable* check, not a mood. Run it over all your Phase 1 code before it goes in the portfolio.

## 5. Recap — automate the boring review

| Check | AST mechanism | Real tool |
|---|---|---|
| Type hints present | `arg.annotation`, `node.returns` | `mypy`, `ruff` |
| Docstrings present | `ast.get_docstring(node)` | `pydocstyle`, `ruff` |
| No hardcoded secrets | walk `Assign` + name match | `gitleaks`, `bandit` |
| Combined gate | run all, print pass/fail | pre-commit + CI |

The `ast` module turns a subjective checklist into an objective gate — the same engine behind every linter you'll use in production. Automate the mechanical checks (hints, docs, secrets) so human review focuses on what machines can't judge: **is the logic right, and is this the simplest design?** That's Phase 1 finished — clean, reviewed, and portfolio-ready.